# 04D1J — LANDFIRE Five-Vintage Consolidation

**Build ID: `04D1J_V1_LANDFIRE_FIVE_VINTAGE_CONSOLIDATION`**

This notebook performs the final LANDFIRE consolidation across all five frozen
retrospective vintages:

| Fire year | LANDFIRE key | Expected units | Expected blocks |
|---|---:|---:|---:|
| 2020 | LF2016R | 48,596 | 450 |
| 2021 | LF2019L | 101,517 | 492 |
| 2022 | LF2020 | 31,232 | 198 |
| 2023 | LF2022 | 10,467 | 111 |
| 2024 | LF2023 | 112,331 | 413 |
| **Total** |  | **304,143** | **1,664** |

## Scope

This notebook **does not recompute LANDFIRE features**.

It combines two already-frozen production families:

1. resolved-vintage production:
   `landfire_resolved_vintage_full_production_v1/feature_shards`
2. archived-vintage production:
   `landfire_archived_vintage_production_v1/feature_shards`

Every source shard is independently validated before consolidation.

## Immutable science contract

- frozen 04D1C V5 feature dictionary;
- 605 frozen LANDFIRE feature fields;
- exact canonical 612-column unit table;
- one row per operational unit;
- fire-year → LANDFIRE-vintage mapping unchanged;
- no feature imputation;
- directional-unavailable fields remain `NaN`;
- source-specific missing-value handling was completed upstream.

The only operation here is **validated row-wise concatenation into canonical
column order**.

## 0. Configuration

In [ ]:
import os  # Configurable project root for the public release.
NOTEBOOK_BUILD_ID = (
    "04D1J_V1_LANDFIRE_FIVE_VINTAGE_CONSOLIDATION"
)

from pathlib import Path
import sys
import subprocess
import importlib.util

PROJECT_ROOT = Path(
    os.environ.get("FIRELINE_PROJECT_ROOT", ".")
)

TERRAIN_UNIT_INDEX_PATH = (
    PROJECT_ROOT / "data" / "processed" /
    "terrain_3dep_production_v1" /
    "terrain_operational_unit_index_v1.parquet"
)

FEATURE_DICTIONARY_PATH = (
    PROJECT_ROOT / "outputs" /
    "landfire_feature_architecture_v5" /
    "LANDFIRE_FEATURE_DICTIONARY_V5.csv"
)

RESOLVED_SHARD_ROOT = (
    PROJECT_ROOT / "data" / "processed" /
    "landfire_resolved_vintage_full_production_v1" /
    "feature_shards"
)

ARCHIVED_SHARD_ROOT = (
    PROJECT_ROOT / "data" / "processed" /
    "landfire_archived_vintage_production_v1" /
    "feature_shards"
)

OUTPUT_ROOT = (
    PROJECT_ROOT / "outputs" /
    "landfire_five_vintage_consolidation_v1"
)

PROCESSED_ROOT = (
    PROJECT_ROOT / "data" / "processed" /
    "landfire_five_vintage_consolidated_v1"
)

CONSOLIDATED_PATH = (
    PROCESSED_ROOT /
    "landfire_features_five_vintage_consolidated_v1.parquet"
)

TEMP_CONSOLIDATED_PATH = (
    CONSOLIDATED_PATH.with_suffix(
        ".parquet.tmp"
    )
)

for p in [
    OUTPUT_ROOT,
    PROCESSED_ROOT,
]:
    p.mkdir(
        parents=True,
        exist_ok=True,
    )

for p in [
    TERRAIN_UNIT_INDEX_PATH,
    FEATURE_DICTIONARY_PATH,
    RESOLVED_SHARD_ROOT,
    ARCHIVED_SHARD_ROOT,
]:
    assert p.exists(), p

VERSION_BY_FIRE_YEAR = {
    2020: "LF2016R",
    2021: "LF2019L",
    2022: "LF2020",
    2023: "LF2022",
    2024: "LF2023",
}

RESOLVED_VERSIONS = [
    "LF2016R",
    "LF2022",
    "LF2023",
]

ARCHIVED_VERSIONS = [
    "LF2019L",
    "LF2020",
]

ALL_VERSIONS = [
    "LF2016R",
    "LF2019L",
    "LF2020",
    "LF2022",
    "LF2023",
]

EXPECTED_VERSION_UNITS = {
    "LF2016R": 48_596,
    "LF2019L": 101_517,
    "LF2020": 31_232,
    "LF2022": 10_467,
    "LF2023": 112_331,
}

EXPECTED_VERSION_BLOCKS = {
    "LF2016R": 450,
    "LF2019L": 492,
    "LF2020": 198,
    "LF2022": 111,
    "LF2023": 413,
}

EXPECTED_TOTAL_UNITS = 304_143
EXPECTED_TOTAL_BLOCKS = 1_664
EXPECTED_V5_FEATURES = 605
EXPECTED_TOTAL_COLUMNS = 612

VALUE_AUDIT_SAMPLE_PER_VERSION = 25
VALUE_AUDIT_SEED = 20260825

# Disk guard for one additional compressed consolidated Parquet.
FREE_SPACE_MULTIPLIER = 1.25
FREE_SPACE_EXTRA_GIB = 2.0

print("=" * 100)
print("RUNNING:", NOTEBOOK_BUILD_ID)
print("=" * 100)
print("Resolved shard root:", RESOLVED_SHARD_ROOT)
print("Archived shard root:", ARCHIVED_SHARD_ROOT)
print("Consolidated output:", CONSOLIDATED_PATH)

## 1. Imports

In [ ]:
required = {
    "pandas": "pandas",
    "numpy": "numpy",
    "pyarrow": "pyarrow",
}

for module, package in required.items():
    if importlib.util.find_spec(
        module
    ) is None:
        subprocess.check_call(
            [
                sys.executable,
                "-m",
                "pip",
                "install",
                package,
            ]
        )

from datetime import datetime, timezone
import hashlib
import json
import math
import os
import shutil
import time

import numpy as np
import pandas as pd

import pyarrow as pa
import pyarrow.parquet as pq

pd.set_option(
    "display.max_columns",
    500,
)
pd.set_option(
    "display.max_rows",
    500,
)
pd.set_option(
    "display.width",
    400,
)

## 2. Load frozen unit universe and exact canonical schema

In [ ]:
unit_index = pd.read_parquet(
    TERRAIN_UNIT_INDEX_PATH
)

feature_dictionary = pd.read_csv(
    FEATURE_DICTIONARY_PATH
)

assert unit_index[
    "unit_uid"
].is_unique

assert feature_dictionary[
    "feature"
].is_unique

V5_FEATURES = list(
    feature_dictionary[
        "feature"
    ].astype(str)
)

V5_FEATURE_SET = set(
    V5_FEATURES
)

assert len(
    V5_FEATURES
) == EXPECTED_V5_FEATURES

PROVENANCE_META_ORDER = [
    "unit_uid",
    "_irwin_norm",
    "IncidentName",
    "Year",
    "_engagement_norm",
    "terrain_block_id",
    "landfire_version_key",
    "landfire_directional_context_valid",
    "approach_vs_line_angle_deg",
    "landfire_source_status",
]

NON_FEATURE_META = [
    c
    for c in PROVENANCE_META_ORDER
    if c not in V5_FEATURE_SET
]

CANONICAL_COLUMNS = (
    NON_FEATURE_META
    + V5_FEATURES
)

assert len(
    CANONICAL_COLUMNS
) == EXPECTED_TOTAL_COLUMNS

assert len(
    CANONICAL_COLUMNS
) == len(
    set(
        CANONICAL_COLUMNS
    )
)

unit_index = unit_index.copy()

unit_index[
    "Year"
] = pd.to_numeric(
    unit_index[
        "Year"
    ],
    errors="raise",
).astype(
    int
)

unit_index[
    "landfire_version_key_expected"
] = unit_index[
    "Year"
].map(
    VERSION_BY_FIRE_YEAR
)

if unit_index[
    "landfire_version_key_expected"
].isna().any():
    raise RuntimeError(
        "Unexpected fire year outside frozen 2020-2024 mapping."
    )

expected_units = unit_index[
    [
        "unit_uid",
        "_irwin_norm",
        "IncidentName",
        "Year",
        "_engagement_norm",
        "terrain_block_id",
        "terrain_block_ix",
        "terrain_block_iy",
        "landfire_version_key_expected",
    ]
].copy()

assert len(
    expected_units
) == EXPECTED_TOTAL_UNITS

version_unit_counts = (
    expected_units.groupby(
        "landfire_version_key_expected",
        as_index=False,
    )
    .agg(
        units=(
            "unit_uid",
            "size",
        ),
        incidents=(
            "_irwin_norm",
            "nunique",
        ),
    )
    .rename(
        columns={
            "landfire_version_key_expected":
            "landfire_version_key",
        }
    )
)

display(
    version_unit_counts
)

for version_key in ALL_VERSIONS:

    observed = int(
        version_unit_counts.loc[
            version_unit_counts[
                "landfire_version_key"
            ] == version_key,
            "units",
        ].iloc[
            0
        ]
    )

    assert observed == EXPECTED_VERSION_UNITS[
        version_key
    ]

print(
    "Frozen V5 features:",
    len(
        V5_FEATURES
    ),
)

print(
    "Non-feature metadata:",
    NON_FEATURE_META,
)

print(
    "Canonical total columns:",
    len(
        CANONICAL_COLUMNS
    ),
)

print(
    "EXPECTED FIVE-VINTAGE UNIT UNIVERSE: PASS"
)

## 3. Build exact expected 1,664-shard manifest

In [ ]:
def safe_token(
    value,
    n=16,
):
    return hashlib.sha1(
        str(
            value
        ).encode(
            "utf-8"
        )
    ).hexdigest()[
        :n
    ]


block_manifest = (
    expected_units.groupby(
        [
            "landfire_version_key_expected",
            "terrain_block_id",
            "terrain_block_ix",
            "terrain_block_iy",
        ],
        as_index=False,
    )
    .agg(
        expected_units=(
            "unit_uid",
            "size",
        ),
        incidents=(
            "_irwin_norm",
            "nunique",
        ),
    )
    .rename(
        columns={
            "landfire_version_key_expected":
            "landfire_version_key",
        }
    )
)

block_manifest[
    "source_family"
] = np.where(
    block_manifest[
        "landfire_version_key"
    ].isin(
        RESOLVED_VERSIONS
    ),
    "resolved",
    "archived",
)

block_manifest[
    "source_root"
] = block_manifest[
    "source_family"
].map({
    "resolved": str(
        RESOLVED_SHARD_ROOT
    ),
    "archived": str(
        ARCHIVED_SHARD_ROOT
    ),
})

block_manifest[
    "shard_name"
] = [
    (
        f"{version_key}__"
        f"{safe_token(block_id)}.parquet"
    )
    for version_key, block_id
    in zip(
        block_manifest[
            "landfire_version_key"
        ],
        block_manifest[
            "terrain_block_id"
        ],
    )
]

block_manifest[
    "shard_path"
] = [
    str(
        (
            RESOLVED_SHARD_ROOT
            if family == "resolved"
            else ARCHIVED_SHARD_ROOT
        ) /
        shard_name
    )
    for family, shard_name
    in zip(
        block_manifest[
            "source_family"
        ],
        block_manifest[
            "shard_name"
        ],
    )
]

block_manifest = block_manifest.sort_values(
    [
        "landfire_version_key",
        "terrain_block_ix",
        "terrain_block_iy",
        "terrain_block_id",
    ]
).reset_index(
    drop=True
)

assert len(
    block_manifest
) == EXPECTED_TOTAL_BLOCKS

version_block_counts = (
    block_manifest.groupby(
        "landfire_version_key",
        as_index=False,
    )
    .agg(
        blocks=(
            "terrain_block_id",
            "size",
        ),
        units=(
            "expected_units",
            "sum",
        ),
    )
)

display(
    version_block_counts
)

for version_key in ALL_VERSIONS:

    row = version_block_counts[
        version_block_counts[
            "landfire_version_key"
        ] == version_key
    ].iloc[
        0
    ]

    assert int(
        row[
            "blocks"
        ]
    ) == EXPECTED_VERSION_BLOCKS[
        version_key
    ]

    assert int(
        row[
            "units"
        ]
    ) == EXPECTED_VERSION_UNITS[
        version_key
    ]

block_manifest.to_csv(
    OUTPUT_ROOT /
    "FIVE_VINTAGE_EXPECTED_SHARD_MANIFEST_V1.csv",
    index=False,
)

version_block_counts.to_csv(
    OUTPUT_ROOT /
    "FIVE_VINTAGE_EXPECTED_WORKLOAD_V1.csv",
    index=False,
)

print(
    "EXPECTED SHARD MANIFEST:",
    len(
        block_manifest
    ),
    "blocks /",
    int(
        block_manifest[
            "expected_units"
        ].sum()
    ),
    "units",
)

## 4. Validate every source shard

For all 1,664 expected block shards the notebook verifies:

- exact file exists;
- Parquet opens;
- row count equals the frozen block-unit count;
- exactly 612 columns;
- exact canonical column **set**;
- records whether source column order is already canonical;
- `unit_uid` is unique within the shard;
- shard `unit_uid` membership exactly equals the frozen expected units for that
  vintage × terrain block.

Column order is normalized only during the final row-wise consolidation.

In [ ]:
expected_uid_by_block = {
    (
        version_key,
        block_id,
    ): set(
        g[
            "unit_uid"
        ]
    )
    for (
        version_key,
        block_id,
    ), g in expected_units.groupby(
        [
            "landfire_version_key_expected",
            "terrain_block_id",
        ],
        sort=False,
    )
}

source_audit_rows = []

audit_start = time.time()

for i, row in block_manifest.iterrows():

    version_key = row[
        "landfire_version_key"
    ]

    block_id = row[
        "terrain_block_id"
    ]

    path = Path(
        row[
            "shard_path"
        ]
    )

    exists = path.exists()

    audit = {
        "landfire_version_key": version_key,
        "terrain_block_id": block_id,
        "source_family": row[
            "source_family"
        ],
        "shard_path": str(
            path
        ),
        "exists": exists,
        "expected_units": int(
            row[
                "expected_units"
            ]
        ),
        "rows": np.nan,
        "columns": np.nan,
        "exact_column_set": False,
        "canonical_column_order": False,
        "unit_uid_unique": False,
        "exact_expected_unit_membership": False,
        "valid": False,
        "reason": None,
        "size_bytes": (
            path.stat().st_size
            if exists
            else np.nan
        ),
    }

    if not exists:
        audit[
            "reason"
        ] = "missing_file"
        source_audit_rows.append(
            audit
        )
        continue

    try:

        meta = pq.read_metadata(
            path
        )

        names = list(
            meta.schema.names
        )

        rows = int(
            meta.num_rows
        )

        audit[
            "rows"
        ] = rows

        audit[
            "columns"
        ] = len(
            names
        )

        audit[
            "exact_column_set"
        ] = (
            len(
                names
            ) == EXPECTED_TOTAL_COLUMNS
            and set(
                names
            ) == set(
                CANONICAL_COLUMNS
            )
        )

        audit[
            "canonical_column_order"
        ] = (
            names
            == CANONICAL_COLUMNS
        )

        uid = pd.read_parquet(
            path,
            columns=[
                "unit_uid",
            ],
        )

        uid_set = set(
            uid[
                "unit_uid"
            ]
        )

        audit[
            "unit_uid_unique"
        ] = (
            len(
                uid
            )
            == len(
                uid_set
            )
        )

        expected_uid_set = (
            expected_uid_by_block[
                (
                    version_key,
                    block_id,
                )
            ]
        )

        audit[
            "exact_expected_unit_membership"
        ] = (
            uid_set
            == expected_uid_set
        )

        row_ok = (
            rows
            == int(
                row[
                    "expected_units"
                ]
            )
        )

        audit[
            "valid"
        ] = bool(
            row_ok
            and audit[
                "exact_column_set"
            ]
            and audit[
                "unit_uid_unique"
            ]
            and audit[
                "exact_expected_unit_membership"
            ]
        )

        audit[
            "reason"
        ] = (
            "valid"
            if audit[
                "valid"
            ]
            else "validation_failure"
        )

    except Exception as exc:

        audit[
            "reason"
        ] = (
            f"{type(exc).__name__}: {exc}"
        )

    source_audit_rows.append(
        audit
    )

    if (
        (i + 1)
        % 100
        == 0
    ):
        print(
            f"Audited {i+1:,} / "
            f"{EXPECTED_TOTAL_BLOCKS:,} shards"
        )


source_shard_audit = pd.DataFrame(
    source_audit_rows
)

display(
    source_shard_audit[
        "valid"
    ].value_counts(
        dropna=False
    )
)

display(
    source_shard_audit.groupby(
        [
            "source_family",
            "landfire_version_key",
        ],
        as_index=False,
    )
    .agg(
        shards=(
            "shard_path",
            "size",
        ),
        valid_shards=(
            "valid",
            "sum",
        ),
        rows=(
            "rows",
            "sum",
        ),
        canonical_order_shards=(
            "canonical_column_order",
            "sum",
        ),
        bytes=(
            "size_bytes",
            "sum",
        ),
    )
)

if not source_shard_audit[
    "valid"
].all():

    display(
        source_shard_audit[
            ~source_shard_audit[
                "valid"
            ]
        ]
    )

    raise RuntimeError(
        "One or more expected source shards failed consolidation preflight."
    )

assert len(
    source_shard_audit
) == EXPECTED_TOTAL_BLOCKS

assert int(
    source_shard_audit[
        "rows"
    ].sum()
) == EXPECTED_TOTAL_UNITS

source_shard_audit.to_csv(
    OUTPUT_ROOT /
    "FIVE_VINTAGE_SOURCE_SHARD_AUDIT_V1.csv",
    index=False,
)

print(
    "ALL 1,664 SOURCE SHARDS: VALID"
)

print(
    "Audit time:",
    f"{(time.time()-audit_start)/60:.2f} min",
)

## 5. Disk-space guard

In [ ]:
source_bytes = int(
    source_shard_audit[
        "size_bytes"
    ].sum()
)

free_bytes = shutil.disk_usage(
    PROCESSED_ROOT
).free

required_free_bytes = int(
    source_bytes
    * FREE_SPACE_MULTIPLIER
    + REQUIRED_FREE_EXTRA_BYTES
) if False else None

# Keep the expression explicit for readability.
required_free_bytes = int(
    source_bytes
    * FREE_SPACE_MULTIPLIER
    + FREE_SPACE_EXTRA_GIB
    * 1024**3
)

disk_qa = pd.DataFrame({
    "metric": [
        "source_shard_bytes",
        "source_shard_gib",
        "free_bytes_before_consolidation",
        "free_gib_before_consolidation",
        "required_free_bytes",
        "required_free_gib",
        "disk_guard_pass",
    ],
    "value": [
        source_bytes,
        source_bytes / 1024**3,
        free_bytes,
        free_bytes / 1024**3,
        required_free_bytes,
        required_free_bytes / 1024**3,
        free_bytes >= required_free_bytes,
    ],
})

display(
    disk_qa
)

if free_bytes < required_free_bytes:
    raise RuntimeError(
        "Insufficient free disk space for safe five-vintage consolidation."
    )

disk_qa.to_csv(
    OUTPUT_ROOT /
    "FIVE_VINTAGE_DISK_QA_V1.csv",
    index=False,
)

print(
    "DISK GUARD: PASS"
)

## 6. Consolidate all 1,664 shards into one canonical Parquet

The writer is streaming:

- one source shard is loaded at a time;
- columns are reordered to the exact canonical 612-column order;
- no feature value is transformed or imputed;
- the first canonical table freezes the Arrow output schema;
- subsequent source shards must cast cleanly into that exact schema;
- output is first written to a temporary file;
- only a fully closed temporary Parquet is atomically promoted to the final
  path.

In [ ]:
TEMP_CONSOLIDATED_PATH.unlink(
    missing_ok=True
)

consolidation_start = time.time()

source_paths = [
    Path(
        p
    )
    for p in source_shard_audit[
        "shard_path"
    ]
]

writer = None
canonical_arrow_schema = None

written_rows = 0

try:

    for i, path in enumerate(
        source_paths,
        start=1,
    ):

        df = pd.read_parquet(
            path
        )

        if set(
            df.columns
        ) != set(
            CANONICAL_COLUMNS
        ):
            raise RuntimeError(
                f"Unexpected schema while consolidating: {path}"
            )

        df = df[
            CANONICAL_COLUMNS
        ].copy()

        table = pa.Table.from_pandas(
            df,
            preserve_index=False,
        )

        if writer is None:

            canonical_arrow_schema = (
                table.schema.remove_metadata()
            )

            writer = pq.ParquetWriter(
                TEMP_CONSOLIDATED_PATH,
                canonical_arrow_schema,
                compression="zstd",
                use_dictionary=True,
                write_statistics=True,
            )

        else:

            table = table.replace_schema_metadata(
                None
            )

            if not table.schema.equals(
                canonical_arrow_schema,
                check_metadata=False,
            ):

                try:
                    table = table.cast(
                        canonical_arrow_schema,
                        safe=True,
                    )

                except Exception as exc:
                    raise RuntimeError(
                        f"Arrow schema mismatch/cast failure for {path}: {exc}"
                    ) from exc

        writer.write_table(
            table
        )

        written_rows += len(
            df
        )

        del df
        del table

        if (
            i % 100
            == 0
        ):
            print(
                f"Wrote {i:,} / "
                f"{EXPECTED_TOTAL_BLOCKS:,} shards | "
                f"{written_rows:,} rows"
            )

finally:

    if writer is not None:
        writer.close()


assert TEMP_CONSOLIDATED_PATH.exists()

temp_meta = pq.read_metadata(
    TEMP_CONSOLIDATED_PATH
)

assert int(
    temp_meta.num_rows
) == EXPECTED_TOTAL_UNITS

assert list(
    temp_meta.schema.names
) == CANONICAL_COLUMNS

# Atomic promotion on same filesystem.
TEMP_CONSOLIDATED_PATH.replace(
    CONSOLIDATED_PATH
)

assert CONSOLIDATED_PATH.exists()

print(
    "Consolidated:",
    CONSOLIDATED_PATH,
)

print(
    "Rows written:",
    written_rows,
)

print(
    "File size GiB:",
    f"{CONSOLIDATED_PATH.stat().st_size/1024**3:.3f}",
)

print(
    "Consolidation time:",
    f"{(time.time()-consolidation_start)/60:.2f} min",
)

## 7. Consolidated-table structural validation

In [ ]:
consolidated_meta = pq.read_metadata(
    CONSOLIDATED_PATH
)

consolidated_schema_names = list(
    consolidated_meta.schema.names
)

assert int(
    consolidated_meta.num_rows
) == EXPECTED_TOTAL_UNITS

assert len(
    consolidated_schema_names
) == EXPECTED_TOTAL_COLUMNS

assert consolidated_schema_names == (
    CANONICAL_COLUMNS
)

id_columns = [
    "unit_uid",
    "Year",
    "terrain_block_id",
    "landfire_version_key",
    "landfire_directional_context_valid",
]

consolidated_ids = pd.read_parquet(
    CONSOLIDATED_PATH,
    columns=id_columns,
)

assert len(
    consolidated_ids
) == EXPECTED_TOTAL_UNITS

assert consolidated_ids[
    "unit_uid"
].nunique() == (
    EXPECTED_TOTAL_UNITS
)

duplicate_uid_rows = (
    len(
        consolidated_ids
    )
    - consolidated_ids[
        "unit_uid"
    ].nunique()
)

expected_uid_set = set(
    expected_units[
        "unit_uid"
    ]
)

consolidated_uid_set = set(
    consolidated_ids[
        "unit_uid"
    ]
)

missing_units = sorted(
    expected_uid_set
    - consolidated_uid_set
)

unexpected_units = sorted(
    consolidated_uid_set
    - expected_uid_set
)

assert duplicate_uid_rows == 0
assert not missing_units
assert not unexpected_units

consolidated_ids[
    "expected_version"
] = (
    pd.to_numeric(
        consolidated_ids[
            "Year"
        ],
        errors="raise",
    )
    .astype(
        int
    )
    .map(
        VERSION_BY_FIRE_YEAR
    )
)

mapping_mismatch = (
    consolidated_ids[
        "landfire_version_key"
    ]
    != consolidated_ids[
        "expected_version"
    ]
)

assert not mapping_mismatch.any()

consolidated_version_counts = (
    consolidated_ids.groupby(
        "landfire_version_key",
        as_index=False,
    )
    .agg(
        rows=(
            "unit_uid",
            "size",
        ),
        unique_units=(
            "unit_uid",
            "nunique",
        ),
        directional_valid_units=(
            "landfire_directional_context_valid",
            "sum",
        ),
    )
)

display(
    consolidated_version_counts
)

for version_key in ALL_VERSIONS:

    row = consolidated_version_counts[
        consolidated_version_counts[
            "landfire_version_key"
        ] == version_key
    ].iloc[
        0
    ]

    assert int(
        row[
            "rows"
        ]
    ) == EXPECTED_VERSION_UNITS[
        version_key
    ]

    assert int(
        row[
            "unique_units"
        ]
    ) == EXPECTED_VERSION_UNITS[
        version_key
    ]

structural_qa = pd.DataFrame({
    "check": [
        "row_count_304143",
        "column_count_612",
        "exact_ordered_canonical_schema",
        "unit_uid_unique",
        "zero_duplicate_uid_rows",
        "zero_missing_expected_units",
        "zero_unexpected_units",
        "fire_year_vintage_mapping_exact",
    ],
    "pass": [
        int(
            consolidated_meta.num_rows
        ) == EXPECTED_TOTAL_UNITS,
        len(
            consolidated_schema_names
        ) == EXPECTED_TOTAL_COLUMNS,
        consolidated_schema_names
        == CANONICAL_COLUMNS,
        consolidated_ids[
            "unit_uid"
        ].nunique()
        == EXPECTED_TOTAL_UNITS,
        duplicate_uid_rows == 0,
        len(
            missing_units
        ) == 0,
        len(
            unexpected_units
        ) == 0,
        not mapping_mismatch.any(),
    ],
})

display(
    structural_qa
)

assert structural_qa[
    "pass"
].all()

consolidated_version_counts.to_csv(
    OUTPUT_ROOT /
    "FIVE_VINTAGE_CONSOLIDATED_VERSION_COUNTS_V1.csv",
    index=False,
)

structural_qa.to_csv(
    OUTPUT_ROOT /
    "FIVE_VINTAGE_STRUCTURAL_QA_V1.csv",
    index=False,
)

print(
    "CONSOLIDATED STRUCTURAL CONTRACT: PASS"
)

## 8. Deterministic value-preservation audit

Consolidation should be a pure row-wise copy/reorder.

To detect accidental transformation, the notebook samples 25 units from each
vintage (125 units total), retrieves their rows from the original source shards
and from the consolidated Parquet, and requires exact equality across all 612
canonical columns.

In [ ]:
rng = np.random.default_rng(
    VALUE_AUDIT_SEED
)

audit_uid_rows = []

for version_key in ALL_VERSIONS:

    candidates = expected_units.loc[
        expected_units[
            "landfire_version_key_expected"
        ] == version_key,
        "unit_uid",
    ].to_numpy()

    n = min(
        VALUE_AUDIT_SAMPLE_PER_VERSION,
        len(
            candidates
        ),
    )

    chosen = rng.choice(
        candidates,
        size=n,
        replace=False,
    )

    for uid in chosen:
        audit_uid_rows.append({
            "landfire_version_key": version_key,
            "unit_uid": uid,
        })


audit_uids = pd.DataFrame(
    audit_uid_rows
)

audit_lookup = audit_uids.merge(
    expected_units[
        [
            "unit_uid",
            "terrain_block_id",
            "landfire_version_key_expected",
        ]
    ],
    on="unit_uid",
    how="left",
    validate="one_to_one",
)

source_sample_frames = []

for (
    version_key,
    block_id,
), g in audit_lookup.groupby(
    [
        "landfire_version_key_expected",
        "terrain_block_id",
    ],
    sort=False,
):

    source_family = (
        "resolved"
        if version_key in RESOLVED_VERSIONS
        else "archived"
    )

    root = (
        RESOLVED_SHARD_ROOT
        if source_family == "resolved"
        else ARCHIVED_SHARD_ROOT
    )

    shard = (
        root /
        f"{version_key}__"
        f"{safe_token(block_id)}.parquet"
    )

    df = pd.read_parquet(
        shard
    )

    df = df[
        df[
            "unit_uid"
        ].isin(
            g[
                "unit_uid"
            ]
        )
    ][
        CANONICAL_COLUMNS
    ].copy()

    source_sample_frames.append(
        df
    )


source_sample = pd.concat(
    source_sample_frames,
    ignore_index=True,
)

sample_uid_list = list(
    audit_uids[
        "unit_uid"
    ]
)

consolidated_sample = pd.read_parquet(
    CONSOLIDATED_PATH,
    filters=[
        (
            "unit_uid",
            "in",
            sample_uid_list,
        )
    ],
)

consolidated_sample = consolidated_sample[
    CANONICAL_COLUMNS
].copy()

assert len(
    source_sample
) == len(
    audit_uids
)

assert len(
    consolidated_sample
) == len(
    audit_uids
)

source_sample = source_sample.sort_values(
    "unit_uid"
).reset_index(
    drop=True
)

consolidated_sample = consolidated_sample.sort_values(
    "unit_uid"
).reset_index(
    drop=True
)

pd.testing.assert_frame_equal(
    source_sample,
    consolidated_sample,
    check_dtype=False,
    check_exact=True,
)

value_audit = pd.DataFrame({
    "metric": [
        "sample_units",
        "sample_per_version_target",
        "all_612_values_exactly_preserved",
    ],
    "value": [
        len(
            audit_uids
        ),
        VALUE_AUDIT_SAMPLE_PER_VERSION,
        True,
    ],
})

display(
    value_audit
)

audit_uids.to_csv(
    OUTPUT_ROOT /
    "FIVE_VINTAGE_VALUE_AUDIT_SAMPLE_UIDS_V1.csv",
    index=False,
)

value_audit.to_csv(
    OUTPUT_ROOT /
    "FIVE_VINTAGE_VALUE_PRESERVATION_QA_V1.csv",
    index=False,
)

print(
    "DETERMINISTIC VALUE-PRESERVATION AUDIT: PASS"
)

## 9. Final consolidation checks

In [ ]:
checks = [
    {
        "check": "exact_five_vintage_set",
        "pass": set(
            consolidated_version_counts[
                "landfire_version_key"
            ]
        ) == set(
            ALL_VERSIONS
        ),
    },
    {
        "check": "expected_source_blocks_1664",
        "pass": len(
            source_shard_audit
        ) == EXPECTED_TOTAL_BLOCKS,
    },
    {
        "check": "all_1664_source_shards_valid",
        "pass": bool(
            source_shard_audit[
                "valid"
            ].all()
        ),
    },
    {
        "check": "source_rows_sum_to_304143",
        "pass": int(
            source_shard_audit[
                "rows"
            ].sum()
        ) == EXPECTED_TOTAL_UNITS,
    },
    {
        "check": "consolidated_rows_304143",
        "pass": int(
            consolidated_meta.num_rows
        ) == EXPECTED_TOTAL_UNITS,
    },
    {
        "check": "consolidated_exact_612_column_schema",
        "pass": (
            consolidated_schema_names
            == CANONICAL_COLUMNS
        ),
    },
    {
        "check": "all_605_frozen_v5_features_present",
        "pass": set(
            V5_FEATURES
        ).issubset(
            set(
                consolidated_schema_names
            )
        )
        and len(
            V5_FEATURES
        ) == EXPECTED_V5_FEATURES,
    },
    {
        "check": "all_units_covered_exactly_once",
        "pass": (
            len(
                consolidated_ids
            ) == EXPECTED_TOTAL_UNITS
            and consolidated_ids[
                "unit_uid"
            ].nunique()
            == EXPECTED_TOTAL_UNITS
            and duplicate_uid_rows == 0
            and len(
                missing_units
            ) == 0
            and len(
                unexpected_units
            ) == 0
        ),
    },
    {
        "check": "frozen_fire_year_to_vintage_mapping_exact",
        "pass": bool(
            not mapping_mismatch.any()
        ),
    },
    {
        "check": "deterministic_sample_values_exactly_preserved",
        "pass": True,
    },
]

checks_df = pd.DataFrame(
    checks
)

display(
    checks_df
)

CONSOLIDATION_PASS = bool(
    checks_df[
        "pass"
    ].all()
)

print()
print(
    "04D1J FIVE-VINTAGE LANDFIRE CONSOLIDATION:",
    (
        "PASS / LANDFIRE FIVE-VINTAGE FEATURES FROZEN"
        if CONSOLIDATION_PASS
        else "REVIEW REQUIRED"
    ),
)

checks_df.to_csv(
    OUTPUT_ROOT /
    "FIVE_VINTAGE_FINAL_CHECKS_V1.csv",
    index=False,
)

## 10. Final report

In [ ]:
def tomd(
    df,
):
    try:
        return df.to_markdown(
            index=False
        )
    except Exception:
        return df.to_string(
            index=False
        )


source_summary = (
    source_shard_audit.groupby(
        [
            "source_family",
            "landfire_version_key",
        ],
        as_index=False,
    )
    .agg(
        shards=(
            "shard_path",
            "size",
        ),
        valid_shards=(
            "valid",
            "sum",
        ),
        rows=(
            "rows",
            "sum",
        ),
        canonical_order_shards=(
            "canonical_column_order",
            "sum",
        ),
        size_bytes=(
            "size_bytes",
            "sum",
        ),
    )
)

source_summary[
    "size_gib"
] = (
    source_summary[
        "size_bytes"
    ]
    / 1024**3
)

report = f"""# 04D1J LANDFIRE Five-Vintage Consolidation

Build: `{NOTEBOOK_BUILD_ID}`

Generated UTC: {datetime.now(timezone.utc).isoformat()}

## Frozen expected workload

{tomd(version_block_counts)}

## Source shard audit summary

{tomd(source_summary)}

## Consolidated table

Path: `{CONSOLIDATED_PATH}`

Rows: `{int(consolidated_meta.num_rows):,}`

Columns: `{len(consolidated_schema_names):,}`

Frozen V5 feature fields: `{len(V5_FEATURES):,}`

File size GiB: `{CONSOLIDATED_PATH.stat().st_size/1024**3:.3f}`

## Version coverage

{tomd(consolidated_version_counts)}

## Structural QA

{tomd(structural_qa)}

## Value-preservation audit

{tomd(value_audit)}

## Final checks

{tomd(checks_df)}

## Decision

`{"PASS / LANDFIRE FIVE-VINTAGE FEATURES FROZEN" if CONSOLIDATION_PASS else "REVIEW REQUIRED"}`

A PASS means all five retrospective LANDFIRE vintages have been consolidated
without recomputing feature values, with exact 304,143-unit coverage and the
canonical frozen 612-column schema.
"""

REPORT_PATH = (
    OUTPUT_ROOT /
    "LANDFIRE_FIVE_VINTAGE_CONSOLIDATION_REPORT_V1.md"
)

REPORT_PATH.write_text(
    report,
    encoding="utf-8",
)

print(
    report
)

print()
print(
    "Saved:",
    REPORT_PATH,
)

## 11. Output inventory

In [ ]:
print("=" * 100)
print("04D1J OUTPUTS")
print("=" * 100)

for p in sorted(
    OUTPUT_ROOT.glob(
        "*"
    )
):
    if p.is_file():
        print(
            f"{p.name:<90}"
            f"{p.stat().st_size/(1024**2):>12.3f} MB"
        )

print()
print(
    "Consolidated dataset:",
    CONSOLIDATED_PATH,
)

print(
    "Size:",
    f"{CONSOLIDATED_PATH.stat().st_size/1024**3:.3f} GiB",
)

print()
print(
    "DONE:",
    NOTEBOOK_BUILD_ID,
)